# Generate Chapters JSON with RAG

Notebook này tích hợp kỹ thuật RAG từ agentic-rag-for-dummies để sinh file JSON theo cấu trúc chapter/lessons từ:
- **Mục lục.md**: Các bài lớn - thứ tự và tên từng đầu mục
- **Các bài lớn/*.md**: Nội dung bài (dùng cho description)
- **Các bài nhỏ/<Bài X>/*.md**: Nội dung từng đầu mục (dùng cho summary và file)

Description và summary được sinh bởi Qwen (Alibaba Intl) theo kiểu RAG.

## 1. Cấu hình (BẮT BUỘC)

- **Gọi trực tiếp Alibaba Intl:** đặt `USE_9ROUTER = False`, nhập `ALIBABA_API_KEY`.
- **Qua 9Router:** đặt `USE_9ROUTER = True`, chạy `npx 9router`, cấu hình Alibaba trong Dashboard và nhập `ROUTER_API_KEY` (hoặc biến môi trường `9ROUTER_API_KEY`).

In [13]:
# ============================================
# Ô 1: CẤU HÌNH - BẮT BUỘC
# ============================================

import os

# Nhập API key của bạn tại đây (dùng khi gọi trực tiếp Alibaba Intl)
ALIBABA_API_KEY = os.environ.get("ALIBABA_API_KEY", "")
ROUTER_API_KEY = os.environ.get("ROUTER_API_KEY", "")

# --- 9Router (proxy tới Alibaba Intl) ---
# Chạy 9Router: npx 9router. Dashboard: http://localhost:20128/dashboard
# Quan trọng: Nếu gặp "No credentials for provider: openai" thì trong Dashboard:
#   - Thêm provider (Alibaba / Qwen hoặc "OpenAI-compatible" với Base URL = https://dashscope-intl.aliyuncs.com/compatible-mode/v1)
#   - Nhập ALIBABA_API_KEY vào provider đó
#   - Tạo Combo (nếu cần) map model "qwen-max" (hoặc tên 9Router dùng) tới provider vừa thêm
# Sau đó lấy API key do 9Router cấp (Settings/API Key) điền vào ROUTER_API_KEY.
USE_9ROUTER = True  # True = gọi qua 9Router, False = gọi trực tiếp Alibaba Intl
ROUTER_BASE_URL = os.environ.get("ROUTER_BASE_URL", "http://localhost:20128/v1")
LLM_MODEL = os.environ.get("LLM_MODEL", "alicode-intl/kimi-k2.5")  # Tên model; qua 9Router có thể cần đổi theo Combo (vd: combo id hoặc "alibaba/qwen-max")

# Cấu hình paths (gốc từ folder textbooks để dễ bàn giao, không phụ thuộc ổ/đường dẫn máy)
# Giả định chạy notebook từ thư mục textbooks/scripts
from pathlib import Path
TEXTBOOKS_ROOT = Path.cwd().parent  # textbooks
BOOK_REL_PATH = "Văn 9/SGK Văn 9 tập 1 - Cánh diều"  # thay đổi theo sách cần generate
BASE_PATH = str(TEXTBOOKS_ROOT / BOOK_REL_PATH)
MUC_LUC_PATH = os.path.join(BASE_PATH, "Các bài lớn", "Mục lục.md")
BAI_LON_PATH = os.path.join(BASE_PATH, "Các bài lớn")
BAI_NHO_PATH = os.path.join(BASE_PATH, "Các bài nhỏ")
OUTPUT_PATH = "./output_chapters.json"

# Endpoint, key và model dùng cho LLM (tự chọn theo USE_9ROUTER)
LLM_BASE_URL = ROUTER_BASE_URL if USE_9ROUTER else None
LLM_API_KEY = ROUTER_API_KEY if USE_9ROUTER else ALIBABA_API_KEY
# LLM_MODEL đã đặt ở trên (qwen-max hoặc tên Combo trong 9Router)

# Kiểm tra API key (chỉ bắt buộc khi gọi trực tiếp Alibaba; qua 9Router có thể để trống nếu 9Router không yêu cầu auth)
if not USE_9ROUTER and not ALIBABA_API_KEY:
    raise ValueError("Vui lòng nhập ALIBABA_API_KEY khi USE_9ROUTER=False!")

print(f"✓ Cấu hình hoàn tất")
print(f"  - LLM: qua 9Router" if USE_9ROUTER else "  - LLM: trực tiếp Alibaba Intl")
print(f"  - Base path: {BASE_PATH}")
print(f"  - Output: {OUTPUT_PATH}")

✓ Cấu hình hoàn tất
  - LLM: qua 9Router
  - Base path: e:\Personal\Projects\Hệ thống Agent Gia sư\textbooks\Văn 9\SGK Văn 9 tập 1 - Cánh diều
  - Output: ./output_chapters.json


## 2. Import Dependencies

In [14]:
# ============================================
# Ô 2: IMPORT DEPENDENCIES
# ============================================

import json
import re
import glob
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional
from dataclasses import dataclass
from openai import OpenAI

print("✓ Dependencies imported")

✓ Dependencies imported


## 3. Định nghĩa Data Classes

In [15]:
# ============================================
# Ô 3: DATA CLASSES
# ============================================

@dataclass
class Lesson:
    """Đại diện cho một bài học nhỏ"""
    lessonTitle: str
    category: str  # Đầu mục cha (từ mục lục), rỗng nếu không có con
    show_type: bool  # True = nội dung quan trọng cần xem, False = có thể bỏ qua
    summary: str
    file: str

    def to_dict(self) -> Dict[str, Any]:
        return {
            "lessonTitle": self.lessonTitle,
            "category": self.category,
            "show_type": self.show_type,
            "summary": self.summary,
            "file": self.file
        }


@dataclass
class Chapter:
    """Đại diện cho một chương/bài lớn"""
    title: str
    description: str
    lessons: List[Lesson]

    def to_dict(self) -> Dict[str, Any]:
        return {
            "chapter": {
                "title": self.title,
                "description": self.description,
                "lessons": [lesson.to_dict() for lesson in self.lessons]
            }
        }

print("✓ Data classes defined")

✓ Data classes defined


## 4. RAG Components - Parent & Child Chunks

In [16]:
# ============================================
# Ô 4: RAG COMPONENTS
# ============================================

class DocumentChunker:
    """Tạo parent và child chunks từ nội dung markdown"""
    
    def __init__(self, max_parent_size: int = 2000, max_child_size: int = 500, overlap: int = 100):
        self.max_parent_size = max_parent_size
        self.max_child_size = max_child_size
        self.overlap = overlap
    
    def create_parent_chunks(self, content: str, source_file: str) -> List[Dict[str, Any]]:
        """Tạo parent chunks - mỗi chunk là một section lớn"""
        # Split by headers
        sections = re.split(r'\n(?=#{1,3}\s)', content)
        
        parent_chunks = []
        current_chunk = ""
        chunk_index = 0
        
        for section in sections:
            section = section.strip()
            if not section:
                continue
            
            # Nếu section quá lớn, split thành nhiều chunks
            if len(section) > self.max_parent_size:
                # Lưu chunk hiện tại nếu có
                if current_chunk:
                    parent_chunks.append({
                        "content": current_chunk.strip(),
                        "source": source_file,
                        "index": chunk_index,
                        "type": "parent"
                    })
                    chunk_index += 1
                    current_chunk = ""
                
                # Split section lớn
                paragraphs = section.split('\n\n')
                for para in paragraphs:
                    if len(current_chunk) + len(para) > self.max_parent_size:
                        if current_chunk:
                            parent_chunks.append({
                                "content": current_chunk.strip(),
                                "source": source_file,
                                "index": chunk_index,
                                "type": "parent"
                            })
                            chunk_index += 1
                            current_chunk = para
                    else:
                        current_chunk += "\n\n" + para if current_chunk else para
            else:
                if len(current_chunk) + len(section) > self.max_parent_size:
                    parent_chunks.append({
                        "content": current_chunk.strip(),
                        "source": source_file,
                        "index": chunk_index,
                        "type": "parent"
                    })
                    chunk_index += 1
                    current_chunk = section
                else:
                    current_chunk += "\n\n" + section if current_chunk else section
        
        # Lưu chunk cuối
        if current_chunk:
            parent_chunks.append({
                "content": current_chunk.strip(),
                "source": source_file,
                "index": chunk_index,
                "type": "parent"
            })
        
        return parent_chunks
    
    def create_child_chunks(self, parent_chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Tạo child chunks từ parent chunks - finer granularity"""
        child_chunks = []
        
        for parent in parent_chunks:
            content = parent["content"]
            parent_index = parent["index"]
            
            # Split thành các đoạn nhỏ hơn
            paragraphs = content.split('\n\n')
            current_child = ""
            child_index = 0
            
            for para in paragraphs:
                para = para.strip()
                if not para:
                    continue
                
                if len(current_child) + len(para) > self.max_child_size:
                    if current_child:
                        child_chunks.append({
                            "content": current_child.strip(),
                            "source": parent["source"],
                            "parent_index": parent_index,
                            "index": child_index,
                            "type": "child"
                        })
                        # Keep overlap
                        words = current_child.split()
                        overlap_text = " ".join(words[-self.overlap//10:]) if len(words) > self.overlap//10 else current_child
                        current_child = overlap_text + "\n\n" + para
                        child_index += 1
                    else:
                        current_child = para
                else:
                    current_child += "\n\n" + para if current_child else para
            
            if current_child:
                child_chunks.append({
                    "content": current_child.strip(),
                    "source": parent["source"],
                    "parent_index": parent_index,
                    "index": child_index,
                    "type": "child"
                })
        
        return child_chunks
    
    def create_chunks_simple(self, content: str, source_file: str, max_chunk_size: int = 1500) -> List[Dict[str, Any]]:
        """Tạo chunks đơn giản - dùng khi content ngắn"""
        chunks = []
        
        # Split by paragraphs first
        paragraphs = content.split('\n\n')
        current_chunk = ""
        chunk_index = 0
        
        for para in paragraphs:
            para = para.strip()
            if not para:
                continue
            
            if len(current_chunk) + len(para) > max_chunk_size:
                if current_chunk:
                    chunks.append({
                        "content": current_chunk.strip(),
                        "source": source_file,
                        "index": chunk_index,
                        "type": "chunk"
                    })
                    chunk_index += 1
                    current_chunk = para
            else:
                current_chunk += "\n\n" + para if current_chunk else para
        
        if current_chunk:
            chunks.append({
                "content": current_chunk.strip(),
                "source": source_file,
                "index": chunk_index,
                "type": "chunk"
            })
        
        return chunks

print("✓ RAG components initialized")

✓ RAG components initialized


## 5. Qwen LLM Client

In [17]:
# ============================================
# Ô 5: QWEN LLM CLIENT (UNLIMITED TOKENS)
# ============================================

class QwenLLM:
    """Client cho Qwen API (Alibaba International) hoặc qua 9Router."""
    
    # Giá trị cao nhất cho max_tokens - pratically unlimited
    # Các model hiện đại thường giới hạn output ~8K tokens
    MAX_TOKENS_UNLIMITED = 8192  # Giới hạn cao nhất của hầu hết các model
    
    def __init__(self, api_key: str, base_url: str | None = None, model: str = "qwen-max"):
        self.client = OpenAI(
            api_key=api_key,
            base_url=base_url or "https://dashscope-intl.aliyuncs.com/compatible-mode/v1"
        )
        self.model = model
    
    def generate_summary(self, chunks: List[Dict[str, Any]], max_length: int = 300, is_description: bool = False) -> str:
        """
        Sinh tóm tắt từ các chunks sử dụng RAG-style generation.
        Kết hợp thông tin từ parent và child chunks.
        """
        # Kết hợp nội dung từ các chunks
        context = self._build_context(chunks)
        
        # Kiểm tra nếu context rỗng
        if not context or len(context.strip()) < 50:
            print(f"    ⚠ Context quá ngắn ({len(context) if context else 0} chars), không gọi LLM")
            return ""
        
        # Điều chỉnh prompt dựa trên loại: description (chapter) hay summary (lesson)
        if is_description:
            prompt = f"""Dựa trên nội dung sau đây, hãy viết một đoạn mô tả CHI TIẾT về bài học (khoảng {max_length} từ).

Nội dung:
{context}

Yêu cầu QUAN TRỌNG:
- Mô tả ĐẦY ĐỦ các nội dung chính: thể loại văn học, tác phẩm, tác giả, kĩ năng được rèn luyện
- Nêu rõ các tác phẩm cụ thể được học (tên tác phẩm, tác giả)
- Liệt kê các kĩ năng: đọc hiểu, viết, nói nghe
- Sử dụng tiếng Việt
- Không tổng quát hóa quá mức, giữ các chi tiết quan trọng
- VIẾT ĐẦY ĐỦ CÂU, hoàn thành tất cả ý nghĩa

Mô tả:"""
        else:
            prompt = f"""Dựa trên nội dung sau đây, hãy viết một đoạn tóm tắt CHI TIẾT (khoảng {max_length} từ).

Nội dung:
{context}

Yêu cầu QUAN TRỌNG:
- Tóm tắt ĐẦY ĐỦ nội dung chính, không bỏ sót ý quan trọng
- Nêu rõ: chủ đề, nội dung cụ thể, ví dụ/ tác phẩm (nếu có)
- Sử dụng tiếng Việt
- Cấu trúc rõ ràng, dễ hiểu
- Không tổng quát hóa thành các cụm từ chung chung
- VIẾT ĐẦY ĐỦ CÂU, hoàn thành tất cả ý nghĩa

Tóm tắt:"""
        
        try:
            print(f"    📝 Gọi LLM với context {len(context)} chars...")
            
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "Bạn là trợ lý giáo dục chuyên tóm tắt nội dung sách giáo khoa. Nhiệm vụ của bạn là tóm tắt CHI TIẾT và ĐẦY ĐỦ, không bỏ sót thông tin quan trọng. Hãy viết cho đến khi hoàn thành tất cả ý nghĩa, không dừng giữa chừng."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.2,
                max_tokens=self.MAX_TOKENS_UNLIMITED  # Dùng giá trị cao nhất
            )
            
            result = response.choices[0].message.content.strip()
            
            # Log thông tin về output
            print(f"    ✓ LLM output: {len(result)} chars, ~{len(result.split())} từ")
            
            # Kiểm tra finish_reason để biết có bị cắt không
            finish_reason = response.choices[0].finish_reason
            if finish_reason == "length":
                print(f"    ⚠ Warning: Output bị cắt do đạt max_tokens limit")
            elif finish_reason == "stop":
                print(f"    ✓ Output hoàn chỉnh (finish_reason: stop)")
            
            return result
            
        except Exception as e:
            print(f"Error generating summary: {e}")
            return ""
    
    def _build_context(self, chunks: List[Dict[str, Any]]) -> str:
        """Xây dựng context từ các chunks - hỗ trợ mọi loại chunk type"""
        context_parts = []
        
        # Xử lý tất cả các loại chunks
        for chunk in chunks:
            chunk_type = chunk.get("type", "unknown")
            content = chunk.get("content", "").strip()
            
            if not content:
                continue
            
            if chunk_type == "parent":
                index = chunk.get("index", 0)
                context_parts.append(f"[Section {index}]\n{content}")
            elif chunk_type == "child":
                parent_idx = chunk.get("parent_index", 0)
                child_idx = chunk.get("index", 0)
                context_parts.append(f"[Detail {parent_idx}.{child_idx}]\n{content}")
            elif chunk_type == "chunk":
                # Từ create_chunks_simple
                index = chunk.get("index", 0)
                context_parts.append(f"[Phần {index + 1}]\n{content}")
            else:
                # Fallback cho các loại khác
                context_parts.append(content)
        
        return "\n\n---\n\n".join(context_parts)

print("✓ Qwen LLM client initialized (unlimited tokens mode)")

✓ Qwen LLM client initialized (unlimited tokens mode)


## 6. Parse Mục lục

In [18]:
# ============================================
# Ô 6: PARSE MỤC LỤC (CẢI TIẾN)
# ============================================

class MucLucParser:
    """Parser cho file Mục lục.md - hỗ trợ cả phần Cuối sách"""
    
    def __init__(self, muc_luc_path: str):
        self.muc_luc_path = muc_luc_path
    
    def parse(self) -> List[Dict[str, Any]]:
        """
        Parse file Mục lục.md và trả về danh sách các bài lớn
        Bao gồm: Bài Mở đầu, Bài 1-5, và Cuối sách (Ôn tập)
        """
        with open(self.muc_luc_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        chapters = []
        
        # Pattern 1: Các bài thông thường (## Bài X. Tên bài)
        chapter_pattern = r'##\s+(Bài\s+[^\n]+)\n+\|\s*#\s*\|\s*Đầu mục\s*\|\s*Trang\s*\|\n\|[-\s|]+\n((?:\|[^\n]+\n)+)'
        
        matches = re.finditer(chapter_pattern, content, re.MULTILINE)
        
        for match in matches:
            chapter_title = match.group(1).strip()
            table_content = match.group(2)
            
            # Parse table rows
            lessons = self._parse_lessons(table_content)
            
            chapters.append({
                "title": chapter_title,
                "lessons": lessons
            })
        
        # Pattern 2: Phần Cuối sách (có thể không có "Bài" trong tên)
        # Tìm phần "Cuối sách" với bảng đầu mục
        cuoi_sach_pattern = r'##\s+(Cuối sách)\s*\n+\|\s*#\s*\|\s*Đầu mục\s*\|\s*Trang\s*\|\n\|[-\s|]+\n((?:\|[^\n]+\n)+)'
        
        cuoi_sach_match = re.search(cuoi_sach_pattern, content, re.MULTILINE)
        if cuoi_sach_match:
            chapter_title = "Bài " + cuoi_sach_match.group(1).strip()  # Đổi thành "Bài Cuối sách"
            table_content = cuoi_sach_match.group(2)
            
            lessons = self._parse_lessons(table_content)
            
            if lessons:  # Chỉ thêm nếu có lessons
                chapters.append({
                    "title": chapter_title,
                    "lessons": lessons
                })
                print(f"✓ Found: {chapter_title} with {len(lessons)} lessons")
        
        # Pattern 3: Các phần đặc biệt khác (nếu có)
        # Tìm các section có bảng nhưng không phải "Bài X"
        special_pattern = r'##\s+([^\n]+)\n+\|\s*#\s*\|\s*Đầu mục\s*\|\s*Trang\s*\|\n\|[-\s|]+\n((?:\|[^\n]+\n)+)'
        
        for match in re.finditer(special_pattern, content, re.MULTILINE):
            chapter_title = match.group(1).strip()
            table_content = match.group(2)
            
            # Bỏ qua nếu đã parse rồi (Bài X hoặc Cuối sách)
            if chapter_title.startswith("Bài ") or chapter_title == "Cuối sách":
                continue
            
            # Bỏ qua "Đầu sách" vì không có bảng đầu mục hợp lệ
            if "Đầu sách" in chapter_title:
                continue
            
            lessons = self._parse_lessons(table_content)
            
            if lessons:
                chapters.append({
                    "title": chapter_title,
                    "lessons": lessons
                })
                print(f"✓ Found special section: {chapter_title}")
        
        return chapters
    
    def _parse_lessons(self, table_content: str) -> List[Dict[str, str]]:
        """Parse các dòng trong bảng thành danh sách lessons. Thêm category: đầu mục cha nếu có con (vd 3.1 -> Đọc hiểu văn bản), rỗng nếu không."""
        rows_data = []  # [{number, title}, ...]
        
        rows = table_content.strip().split('\n')
        for row in rows:
            row = row.strip()
            if not row or row.startswith('|---'):
                continue
            parts = [p.strip() for p in row.split('|') if p.strip()]
            if len(parts) < 2:
                continue
            lesson_title = parts[1]
            lesson_title = re.sub(r'\s*\(tr\.\s*\d+\)', '', lesson_title)
            lesson_title = re.sub(r'^[—\s]+', '', lesson_title)
            if not lesson_title or lesson_title in ('Đầu mục', 'Trang'):
                continue
            num = parts[0] if parts else ""
            rows_data.append({"number": num, "title": lesson_title})
        
        # Số nào có dòng con (vd 3.1, 3.2) thì là cha -> map number -> title cha
        parent_titles = {}
        for r in rows_data:
            n = r["number"]
            if "." in n:
                parent_num = n.split(".")[0]
                # Tìm title của cha (đã có trong rows_data)
                for r2 in rows_data:
                    if r2["number"] == parent_num:
                        parent_titles[n] = r2["title"]
                        break
        
        lessons = []
        for r in rows_data:
            category = parent_titles.get(r["number"], "")
            lessons.append({
                "title": r["title"],
                "number": r["number"],
                "category": category
            })
        return lessons

# Test parser
parser = MucLucParser(MUC_LUC_PATH)
chapters_structure = parser.parse()
print(f"\n✓ Parsed {len(chapters_structure)} chapters total:")
for ch in chapters_structure:
    print(f"  - {ch['title']}: {len(ch['lessons'])} lessons")

✓ Found: Bài Cuối sách with 3 lessons

✓ Parsed 7 chapters total:
  - Bài Mở đầu: 6 lessons
  - Bài 1. Thơ và thơ song thất lục bát: 14 lessons
  - Bài 2. Truyện thơ Nôm: 13 lessons
  - Bài 3. Văn bản thông tin: 13 lessons
  - Bài 4. Truyện ngắn: 14 lessons
  - Bài 5. Nghị luận xã hội: 13 lessons
  - Bài Cuối sách: 3 lessons


## 7. File Manager

In [19]:
# ============================================
# Ô 7: FILE MANAGER (CẢI TIẾN)
# ============================================

class FileManager:
    """Quản lý mapping giữa tên bài và file path - với fuzzy matching"""
    
    def __init__(self, bai_lon_path: str, bai_nho_path: str):
        self.bai_lon_path = Path(bai_lon_path)
        self.bai_nho_path = Path(bai_nho_path)
        self.bai_lon_files = self._scan_bai_lon()
        self.bai_nho_folders = self._scan_bai_nho()
        self._build_lesson_file_cache()
    
    def _scan_bai_lon(self) -> Dict[str, Path]:
        """Scan các file trong Các bài lớn"""
        files = {}
        for md_file in self.bai_lon_path.glob('*.md'):
            if md_file.name == 'Mục lục.md':
                continue
            key = md_file.stem
            files[key] = md_file
        return files
    
    def _scan_bai_nho(self) -> Dict[str, Path]:
        """Scan các folder trong Các bài nhỏ"""
        folders = {}
        for folder in self.bai_nho_path.iterdir():
            if folder.is_dir():
                key = folder.name
                folders[key] = folder
        return folders
    
    def _build_lesson_file_cache(self):
        """Build cache của tất cả lesson files để tìm kiếm nhanh"""
        self.lesson_file_cache = {}  # {bai_nho_folder_name: {normalized_title: file_path}}
        
        for folder_name, folder_path in self.bai_nho_folders.items():
            self.lesson_file_cache[folder_name] = {}
            for md_file in folder_path.glob('*.md'):
                # Lưu với nhiều key khác nhau để tìm kiếm
                normalized = self._normalize_for_compare(md_file.stem)
                self.lesson_file_cache[folder_name][normalized] = md_file
                
                # Cũng lưu với key là tên gốc
                self.lesson_file_cache[folder_name][md_file.stem.lower()] = md_file
    
    def get_bai_lon_file(self, chapter_title: str) -> Path:
        """Tìm file bài lớn tương ứng với chapter title"""
        normalized = chapter_title.replace('.', ' -')
        
        # Tìm exact match trước
        if normalized in self.bai_lon_files:
            return self.bai_lon_files[normalized]
        
        # Tìm partial match
        normalized_compare = self._normalize_for_compare(normalized)
        for key, path in self.bai_lon_files.items():
            if self._normalize_for_compare(key) == normalized_compare:
                return path
        
        # Fuzzy match - tìm key có độ tương đồng cao nhất
        best_match = None
        best_score = 0
        for key in self.bai_lon_files.keys():
            score = self._calculate_similarity(normalized_compare, self._normalize_for_compare(key))
            if score > best_score and score > 0.7:  # Ngưỡng 70%
                best_score = score
                best_match = key
        
        if best_match:
            print(f"    ⚡ Fuzzy match bài lớn: '{chapter_title}' -> '{best_match}' (score: {best_score:.2f})")
            return self.bai_lon_files[best_match]
        
        return None
    
    def get_bai_nho_folder(self, chapter_title: str) -> Path:
        """Tìm folder bài nhỏ tương ứng với chapter title"""
        normalized = chapter_title.replace('.', ' -')
        
        if normalized in self.bai_nho_folders:
            return self.bai_nho_folders[normalized]
        
        # Tìm partial match
        normalized_compare = self._normalize_for_compare(normalized)
        for key, path in self.bai_nho_folders.items():
            if self._normalize_for_compare(key) == normalized_compare:
                return path
        
        # Fuzzy match
        best_match = None
        best_score = 0
        for key in self.bai_nho_folders.keys():
            score = self._calculate_similarity(normalized_compare, self._normalize_for_compare(key))
            if score > best_score and score > 0.7:
                best_score = score
                best_match = key
        
        if best_match:
            print(f"    ⚡ Fuzzy match bài nhỏ folder: '{chapter_title}' -> '{best_match}' (score: {best_score:.2f})")
            return self.bai_nho_folders[best_match]
        
        return None
    
    def get_lesson_file(self, bai_nho_folder: Path, lesson_title: str) -> Path:
        """Tìm file bài nhỏ tương ứng với lesson title - với nhiều chiến lược"""
        if not bai_nho_folder or not bai_nho_folder.exists():
            return None
        
        folder_name = bai_nho_folder.name
        if folder_name not in self.lesson_file_cache:
            return None
        
        cache = self.lesson_file_cache[folder_name]
        normalized_title = self._normalize_for_compare(lesson_title)
        
        # Chiến lược 1: Exact match với normalized
        if normalized_title in cache:
            return cache[normalized_title]
        
        # Chiến lược 2: Tìm trong keys
        for key in cache.keys():
            if normalized_title == key or normalized_title in key or key in normalized_title:
                return cache[key]
        
        # Chiến lược 3: Tách phần sau dấu "-" (nếu có)
        if ' - ' in lesson_title:
            parts = lesson_title.split(' - ', 1)
            if len(parts) == 2:
                # Thử tìm với phần sau
                second_part = self._normalize_for_compare(parts[1])
                for key in cache.keys():
                    if second_part in key:
                        return cache[key]
                
                # Thử tìm với phần đầu
                first_part = self._normalize_for_compare(parts[0])
                for key in cache.keys():
                    if first_part in key:
                        return cache[key]
        
        # Chiến lược 4: Fuzzy matching
        best_match = None
        best_score = 0
        for key, file_path in cache.items():
            score = self._calculate_similarity(normalized_title, key)
            if score > best_score and score > 0.6:  # Ngưỡng 60%
                best_score = score
                best_match = file_path
        
        if best_match:
            print(f"    ⚡ Fuzzy match lesson: '{lesson_title[:50]}...' -> '{best_match.name}' (score: {best_score:.2f})")
        
        return best_match
    
    def _normalize_for_compare(self, text: str) -> str:
        """Normalize text để so sánh - nâng cao"""
        text = text.lower()
        # Bỏ dấu tiếng Việt để so sánh dễ hơn
        text = self._remove_vietnamese_accents(text)
        text = re.sub(r'[^\w\s]', ' ', text)  # Thay dấu câu bằng space
        text = re.sub(r'\s+', ' ', text)  # Normalize whitespace
        return text.strip()
    
    def _remove_vietnamese_accents(self, text: str) -> str:
        """Bỏ dấu tiếng Việt"""
        accents = {
            'àáảãạâầấẩẫậăằắẳẵặ': 'a',
            'èéẻẽẹêềếểễệ': 'e',
            'ìíỉĩị': 'i',
            'òóỏõọôồốổỗộơờớởỡợ': 'o',
            'ùúủũụưừứửữự': 'u',
            'ỳýỷỹỵ': 'y',
            'đ': 'd'
        }
        for accented, normal in accents.items():
            for char in accented:
                text = text.replace(char, normal)
        return text
    
    def _calculate_similarity(self, s1: str, s2: str) -> float:
        """Tính độ tương đồng giữa 2 chuỗi (0-1)"""
        # Simple Jaccard similarity
        set1 = set(s1.split())
        set2 = set(s2.split())
        
        if not set1 or not set2:
            return 0.0
        
        intersection = len(set1 & set2)
        union = len(set1 | set2)
        
        return intersection / union if union > 0 else 0.0
    
    def match_lesson_entry(self, lessons_data: List[Dict], file_stem: str) -> Optional[Dict]:
        """Tìm đầu mục trong Mục lục khớp với tên file (stem). Trả về dict {title, number, category} hoặc None."""
        if not lessons_data or not file_stem:
            return None
        norm_stem = self._normalize_for_compare(file_stem)
        for le in lessons_data:
            if self._normalize_for_compare(le.get("title", "")) == norm_stem:
                return le
        best = None
        best_score = 0.6
        for le in lessons_data:
            score = self._calculate_similarity(norm_stem, self._normalize_for_compare(le.get("title", "")))
            if score > best_score:
                best_score = score
                best = le
        return best
    
    def read_file_content(self, file_path: Path) -> str:
        """Đọc nội dung file"""
        if not file_path or not file_path.exists():
            return ""
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                return f.read()
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
            return ""
    
    def list_available_lessons(self, bai_nho_folder: Path) -> List[str]:
        """Liệt kê tất cả lesson files có trong folder"""
        if not bai_nho_folder or not bai_nho_folder.exists():
            return []
        
        lessons = []
        for md_file in bai_nho_folder.glob('*.md'):
            lessons.append(md_file.stem)
        return lessons

# Test file manager
file_manager = FileManager(BAI_LON_PATH, BAI_NHO_PATH)
print(f"✓ Found {len(file_manager.bai_lon_files)} bài lớn files")
print(f"✓ Found {len(file_manager.bai_nho_folders)} bài nhỏ folders")
print(f"✓ Cached {sum(len(files) for files in file_manager.lesson_file_cache.values())} lesson files")

# Test fuzzy matching
test_chapter = "Bài 1. Thơ và thơ song thất lục bát"
bai_lon = file_manager.get_bai_lon_file(test_chapter)
if bai_lon:
    print(f"✓ Test match: '{test_chapter}' -> {bai_lon.name}")
else:
    print(f"✗ Test match failed: '{test_chapter}'")

✓ Found 9 bài lớn files
✓ Found 7 bài nhỏ folders
✓ Cached 118 lesson files
✓ Test match: 'Bài 1. Thơ và thơ song thất lục bát' -> Bài 1 - Thơ và thơ song thất lục bát.md


## 8. Main Generator

In [20]:
# ============================================
# Ô 8: MAIN GENERATOR (CẢI TIẾN - B+C)
# ============================================

class ChapterJSONGenerator:
    """Generator chính cho chapters JSON - với logging chi tiết"""
    
    def __init__(self, api_key: str, base_path: str, base_url: str | None = None, model: str = "qwen-max"):
        self.llm = QwenLLM(api_key, base_url=base_url, model=model)
        self.chunker = DocumentChunker()
        self.muc_luc_parser = MucLucParser(os.path.join(base_path, "Các bài lớn", "Mục lục.md"))
        self.file_manager = FileManager(
            os.path.join(base_path, "Các bài lớn"),
            os.path.join(base_path, "Các bài nhỏ")
        )
        self.base_path = base_path
        self.stats = {
            "chapters_processed": 0,
            "lessons_processed": 0,
            "lessons_with_file": 0,
            "lessons_with_summary": 0,
            "fuzzy_matches": 0,
            "missing_files": []
        }
    
    def generate(self) -> List[Dict[str, Any]]:
        """Generate JSON cho tất cả chapters"""
        chapters_structure = self.muc_luc_parser.parse()
        result = []
        
        print(f"\n{'='*60}")
        print(f"Generating JSON for {len(chapters_structure)} chapters...")
        print(f"{'='*60}\n")
        
        for i, chapter_data in enumerate(chapters_structure, 1):
            chapter_title = chapter_data["title"]
            print(f"\n[{i}/{len(chapters_structure)}] 📚 {chapter_title}")
            print("-" * 60)
            
            chapter = self._process_chapter(chapter_title, chapter_data["lessons"])
            if chapter:
                result.append(chapter.to_dict())
                self.stats["chapters_processed"] += 1
        
        # Print summary
        self._print_summary()
        
        return result
    
    def _process_chapter(self, chapter_title: str, lessons_data: List[Dict]) -> Chapter:
        """Xử lý một chapter"""
        
        # 1. Lấy nội dung bài lớn cho description
        bai_lon_file = self.file_manager.get_bai_lon_file(chapter_title)
        description = ""
        
        if bai_lon_file:
            print(f"  📖 Bài lớn: {bai_lon_file.name}")
            content = self.file_manager.read_file_content(bai_lon_file)
            if content:
                # DÙNG PARENT-CHILD CHUNKS (B)
                parent_chunks = self.chunker.create_parent_chunks(content, str(bai_lon_file))
                child_chunks = self.chunker.create_child_chunks(parent_chunks)
                all_chunks = parent_chunks + child_chunks
                
                if all_chunks:
                    # Truyền is_description=True để dùng prompt phù hợp (C)
                    description = self.llm.generate_summary(all_chunks, max_length=300, is_description=True)
                    print(f"  ✓ Description: {len(description)} chars")
                else:
                    print(f"  ⚠ Không tạo được chunks từ bài lớn")
                    description = self._extract_first_paragraph(content, max_length=500)
            else:
                print(f"  ⚠ File bài lớn rỗng")
        else:
            print(f"  ⚠ Không tìm thấy file bài lớn cho: {chapter_title}")
            self.stats["missing_files"].append(f"Bài lớn: {chapter_title}")
        
        if not description:
            description = f"Nội dung bài {chapter_title}"
        
        # 2. Truy cứu file trước: lấy danh sách file .md trong folder bài nhỏ
        bai_nho_folder = self.file_manager.get_bai_nho_folder(chapter_title)
        lesson_files = []
        if bai_nho_folder and bai_nho_folder.exists():
            print(f"  📁 Bài nhỏ folder: {bai_nho_folder.name}")
            lesson_files = sorted(bai_nho_folder.glob('*.md'))
            print(f"  📄 Có {len(lesson_files)} file lessons (nguồn lesson)")
        else:
            print(f"  ⚠ Không tìm thấy folder bài nhỏ cho: {chapter_title}")
            self.stats["missing_files"].append(f"Bài nhỏ folder: {chapter_title}")
        
        # 3. Với mỗi file, ghép Mục lục (title, category) rồi tạo lesson — file luôn có
        lessons = []
        for file_path in lesson_files:
            stem = file_path.stem
            matched = self.file_manager.match_lesson_entry(lessons_data, stem)
            if matched:
                lesson_data = {"title": matched.get("title", stem), "category": matched.get("category", "") or ""}
            else:
                lesson_data = {"title": stem, "category": ""}
            lesson = self._process_lesson_from_file(file_path, lesson_data)
            if lesson:
                lessons.append(lesson)
                self.stats["lessons_processed"] += 1
                self.stats["lessons_with_file"] += 1
                if lesson.summary and not lesson.summary.startswith("Nội dung"):
                    self.stats["lessons_with_summary"] += 1
        
        print(f"  ✓ Lessons: {len(lessons)} (từ {len(lesson_files)} file)")
        
        return Chapter(
            title=chapter_title,
            description=description,
            lessons=lessons
        )
    
    def _compute_show_type(self, lesson_data: Dict[str, Any]) -> bool:
        """True = nội dung quan trọng cần xem; False = có thể bỏ qua mà vẫn hiểu bài."""
        category = (lesson_data.get("category") or "").strip().lower()
        title = (lesson_data.get("title") or "").strip().lower()
        # Nội dung quan trọng: đọc hiểu văn bản, thực hành đọc hiểu, viết, nói và nghe
        show_true_keywords = (
            "đọc hiểu văn bản", "thực hành đọc hiểu", "viết", "nói và nghe"
        )
        if category:
            for kw in show_true_keywords:
                if kw in category:
                    return True
        for kw in show_true_keywords:
            if kw in title:
                return True
        return False
    
    def _process_lesson_from_file(self, file_path: Path, lesson_data: Dict[str, Any]) -> Lesson:
        """Tạo lesson từ file (file luôn tồn tại). lesson_data có title, category từ Mục lục hoặc mặc định."""
        lesson_title = lesson_data.get("title", file_path.stem)
        category = lesson_data.get("category", "") or ""
        show_type = self._compute_show_type(lesson_data)
        
        relative_path = str(file_path.relative_to(self.base_path)).replace('\\', '/')
        
        content = self.file_manager.read_file_content(file_path)
        summary = ""
        if content:
            parent_chunks = self.chunker.create_parent_chunks(content, str(file_path))
            child_chunks = self.chunker.create_child_chunks(parent_chunks)
            all_chunks = parent_chunks + child_chunks
            if all_chunks:
                summary = self.llm.generate_summary(all_chunks, max_length=250, is_description=False)
            else:
                summary = self._extract_first_paragraph(content, max_length=400)
        if not summary:
            summary = f"Nội dung {lesson_title}"
        
        summary_status = "✓" if not summary.startswith("Nội dung") else "~"
        print(f"    [✓{summary_status}] {lesson_title[:60]}{'...' if len(lesson_title) > 60 else ''}")
        
        return Lesson(
            lessonTitle=lesson_title,
            category=category,
            show_type=show_type,
            summary=summary,
            file=relative_path
        )
    
    def _extract_first_paragraph(self, content: str, max_length: int = 300) -> str:
        """Trích xuất đoạn đầu tiên từ content nếu LLM fail"""
        # Remove markdown headers
        text = re.sub(r'^#+\s+', '', content, flags=re.MULTILINE)
        # Split by paragraphs
        paragraphs = text.split('\n\n')
        for para in paragraphs:
            para = para.strip()
            if len(para) > 50:  # Đoạn có nghĩa
                return para[:max_length] + "..." if len(para) > max_length else para
        return content[:max_length] if content else ""
    
    def _print_summary(self):
        """In tổng kết"""
        print(f"\n{'='*60}")
        print("📊 GENERATION SUMMARY")
        print(f"{'='*60}")
        print(f"  Chapters processed: {self.stats['chapters_processed']}")
        print(f"  Lessons processed: {self.stats['lessons_processed']}")
        print(f"  Lessons with file: {self.stats['lessons_with_file']} ({self.stats['lessons_with_file']/max(self.stats['lessons_processed'],1)*100:.1f}%)")
        print(f"  Lessons with LLM summary: {self.stats['lessons_with_summary']} ({self.stats['lessons_with_summary']/max(self.stats['lessons_processed'],1)*100:.1f}%)")
        
        if self.stats['missing_files']:
            print(f"\n  ⚠ Missing files ({len(self.stats['missing_files'])}):")
            for missing in self.stats['missing_files'][:10]:
                print(f"    - {missing}")
            if len(self.stats['missing_files']) > 10:
                print(f"    ... và {len(self.stats['missing_files']) - 10} file khác")

print("✓ Chapter JSON Generator initialized (B+C)")

✓ Chapter JSON Generator initialized (B+C)


## 9. Run Generation

In [21]:
# ============================================
# Ô 9: RUN GENERATION
# ============================================

# Khởi tạo generator (dùng LLM_* từ ô cấu hình)
generator = ChapterJSONGenerator(LLM_API_KEY, BASE_PATH, base_url=LLM_BASE_URL, model=LLM_MODEL)

# Generate JSON
chapters_json = generator.generate()

print(f"\n\n✓ Generated {len(chapters_json)} chapters")

✓ Found: Bài Cuối sách with 3 lessons

Generating JSON for 7 chapters...


[1/7] 📚 Bài Mở đầu
------------------------------------------------------------
  📖 Bài lớn: Bài mở đầu.md
    📝 Gọi LLM với context 30531 chars...
    ✓ LLM output: 3796 chars, ~764 từ
    ✓ Output hoàn chỉnh (finish_reason: stop)
  ✓ Description: 3796 chars
  📁 Bài nhỏ folder: Bài mở đầu
  📄 Có 3 file lessons (nguồn lesson)
    📝 Gọi LLM với context 3616 chars...
    ✓ LLM output: 1501 chars, ~340 từ
    ✓ Output hoàn chỉnh (finish_reason: stop)
    [✓✓] Cấu trúc sách Ngữ văn 9
    📝 Gọi LLM với context 26339 chars...
    ✓ LLM output: 3841 chars, ~815 từ
    ✓ Output hoàn chỉnh (finish_reason: stop)
    [✓✓] Nội dung sách Ngữ văn 9
    📝 Gọi LLM với context 450 chars...
    ✓ LLM output: 1331 chars, ~298 từ
    ✓ Output hoàn chỉnh (finish_reason: stop)
    [✓✓] Yêu cầu cần đạt
  ✓ Lessons: 3 (từ 3 file)

[2/7] 📚 Bài 1. Thơ và thơ song thất lục bát
------------------------------------------------------------
 

## 10. Save Output

In [22]:
# ============================================
# Ô 10: SAVE OUTPUT
# ============================================

# Lưu ra file JSON
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(chapters_json, f, ensure_ascii=False, indent=2)

print(f"✓ Saved to: {OUTPUT_PATH}")
print(f"✓ Total chapters: {len(chapters_json)}")

✓ Saved to: ./output_chapters.json
✓ Total chapters: 7


## 11. Preview Result

In [23]:
# ============================================
# Ô 11: PREVIEW RESULT
# ============================================

# Hiển thị preview của chapter đầu tiên
if chapters_json:
    print("=" * 60)
    print("PREVIEW - First Chapter:")
    print("=" * 60)
    print(json.dumps(chapters_json[0], ensure_ascii=False, indent=2))
    
    # Thống kê
    print("\n" + "=" * 60)
    print("STATISTICS:")
    print("=" * 60)
    total_lessons = sum(len(ch["chapter"]["lessons"]) for ch in chapters_json)
    print(f"Total chapters: {len(chapters_json)}")
    print(f"Total lessons: {total_lessons}")
    print(f"Avg lessons per chapter: {total_lessons / len(chapters_json):.1f}")
    
    # Phân loại theo category và show_type
    category_counts = {}
    show_true, show_false = 0, 0
    for ch in chapters_json:
        for lesson in ch["chapter"]["lessons"]:
            c = lesson.get("category", "") or "(none)"
            category_counts[c] = category_counts.get(c, 0) + 1
            if lesson.get("show_type", False):
                show_true += 1
            else:
                show_false += 1
    
    print("\nLesson category distribution:")
    for c, count in sorted(category_counts.items(), key=lambda x: -x[1]):
        print(f"  - {c}: {count}")
    print(f"\nshow_type: True={show_true}, False={show_false}")

PREVIEW - First Chapter:
{
  "chapter": {
    "title": "Bài Mở đầu",
    "description": "Bài Mở đầu \"Nội dung và cấu trúc sách Ngữ văn 9\" cung cấp cái nhìn tổng quan toàn diện về chương trình học, giúp học sinh nắm vững những nội dung chính, cấu trúc sách và cách sử dụng sách hiệu quả. Về nội dung học đọc, sách hướng dẫn đọc hiểu năm thể loại văn bản với nhiều tác phẩm tiêu biểu. Thể loại truyện bao gồm truyện thơ Nôm với \"Cảnh ngày xuân\", \"Kiều ở lầu Ngưng Bích\" (trích Truyện Kiều - Nguyễn Du), \"Lục Vân Tiên cứu Kiều Nguyệt Nga\", \"Lục Vân Tiên gặp nạn\" (trích Truyện Lục Vân Tiên - Nguyễn Đình Chiểu); truyện ngắn với \"Làng\" (Kim Lân), \"Ông lão bên chiếc cầu\" (Ernest Hemingway), \"Chiếc lược ngà\" (Nguyễn Quang Sáng), \"Chiếc lá cuối cùng\" (O. Henry), \"Những con cá cờ\" (Trần Đức Tiến), \"Người thứ bảy\" (Murakami), \"Chị tôi\" (Nguyễn Thị Thu Huệ); truyện truyền kì với \"Chuyện người con gái Nam Xương\" (Nguyễn Dữ), \"Dế chọi\" (Bồ Tùng Linh); và truyện trinh thám với \

## 12. Export Individual Chapter Files (Optional)

In [24]:
# ============================================
# Ô 12: EXPORT INDIVIDUAL CHAPTER FILES (OPTIONAL)
# ============================================

# Nếu muốn export mỗi chapter ra file riêng
EXPORT_INDIVIDUAL = False
INDIVIDUAL_OUTPUT_DIR = "./chapter_jsons"

if EXPORT_INDIVIDUAL and chapters_json:
    import os
    os.makedirs(INDIVIDUAL_OUTPUT_DIR, exist_ok=True)
    
    for i, chapter_data in enumerate(chapters_json):
        chapter = chapter_data["chapter"]
        # Tạo tên file từ title
        safe_title = re.sub(r'[^\w\s-]', '', chapter["title"]).strip().replace(' ', '_')
        filename = f"{i+1:02d}_{safe_title}.json"
        filepath = os.path.join(INDIVIDUAL_OUTPUT_DIR, filename)
        
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(chapter_data, f, ensure_ascii=False, indent=2)
        
        print(f"✓ Exported: {filename}")
    
    print(f"\n✓ All chapters exported to: {INDIVIDUAL_OUTPUT_DIR}")